# Q-Commerce Stockout Prediction Pipeline
**Project:** Advanced Machine Learning for Business Transformation (AMLBT)
**Team:** Nishant Khandelwal, Anwesha Banerjee, Anisha Saha

This notebook executes the data preprocessing, model training, and evaluation for predicting inventory failure.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# 1. Load the dataset from the root directory
df = pd.read_csv('inventory_optimization.csv')

# Print columns to verify mappings
print("Dataset Columns:", df.columns.tolist())

# 2. Define the target variable
target_col = 'stockout_occurred' 
if target_col not in df.columns:
    target_col = df.columns[-1] # Fallback to the last column if naming differs

df = df.dropna(subset=[target_col])

# 3. Identify and impute numerical features
num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
if target_col in num_cols:
    num_cols.remove(target_col)

df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# 4. Scale continuous features for the Logistic Regression baseline
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

print("\nData Preprocessing Complete. Current shape:", df.shape)
display(df.head())

In [ ]:
# Separate features and target variable
X = df.drop(columns=[target_col])
y = df[target_col]

# Convert categorical variables to dummy variables if any remain
X = pd.get_dummies(X, drop_first=True)

# 70-30 Train-Test Split (stratified to handle class imbalance)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

In [ ]:
# Initialize models
models = {
    "Logistic Regression (Baseline)": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "XGBoost": xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss')
}

# Train and evaluate each model
for name, model in models.items():
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    preds = model.predict(X_test)
    
    # Evaluate
    precision = precision_score(y_test, preds, zero_division=0)
    recall = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    
    print(f"--- {name} ---")
    print(f"Precision : {precision:.2f}")
    print(f"Recall    : {recall:.2f}")
    print(f"F1-Score  : {f1:.2f}\n")

In [ ]:
# Extract XGBoost model from the dictionary
xgb_model = models["XGBoost"]

# Plot feature importance
plt.figure(figsize=(8, 5))
xgb.plot_importance(xgb_model, importance_type='weight', max_num_features=5, height=0.5, show_values=False)
plt.title('XGBoost Feature Importance: Predicting Stockouts')
plt.ylabel('Features')
plt.xlabel('F-Score (Weight)')
plt.tight_layout()
plt.show()